# PDF support

Send a PDF in a user message as a **document block**, next to a text block.
Compared with image support, change `image` to `document` and `image/png` to
`application/pdf`. Base64 encodes the file for transport; Claude reads the PDF.

The example is the course's Wikipedia article about Earth, saved as [`earth.pdf`](./earth.pdf).
No local text extraction or Files API upload is needed.

## Setup

The same client and helpers as in image support.

In [4]:
import base64
from pathlib import Path

from anthropic import Anthropic
from anthropic.types import Message
from dotenv import load_dotenv

load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None, max_tokens=1000):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")


## Read the PDF

Use `document` and `application/pdf`. The data field contains the base64 string,
without a `data:` URL prefix.

In [5]:
pdf_path = Path("earth.pdf")
file_bytes = base64.standard_b64encode(pdf_path.read_bytes()).decode("utf-8")

document = {
    "type": "document",
    "source": {
        "type": "base64",
        "media_type": "application/pdf",
        "data": file_bytes,
    },
}
print(f"{pdf_path.name}: {pdf_path.stat().st_size:,} bytes")

earth.pdf: 886,832 bytes


## Summarize the document

The lesson's prompt. The document and the question go in the same user message.
Print only the text and usage, keeping the large base64 payload out of the output.

In [6]:
messages = []
add_user_message(messages, [
    document,
    {"type": "text", "text": "Summarize the document in one sentence"},
])

response = chat(messages)
print(text_from_message(response))
print(f"\n[stop_reason: {response.stop_reason}, "
      f"input tokens: {response.usage.input_tokens}, "
      f"output tokens: {response.usage.output_tokens}]")

# Summary

Earth is the third planet from the Sun and the only known astronomical object that harbors life, sustained by its oceans covering 70.8% of its surface, a dynamic atmosphere, and a magnetic field that protects it from solar radiation.

[stop_reason: end_turn, input tokens: 9625, output tokens: 56]


The saved response below the code shows whether this document block works through
our configured provider. This is a compatibility check on one PDF, not a test of
all PDF features or extraction accuracy.

Next lesson: citations.